In [7]:
import torch
from transformers import GPT2LMHeadModel, GPT2Tokenizer
from typing import List

# 加载 Reward Model 和 Tokenizer
model_name = "/home/jovyan/notebook/f-divergence-dpo/ppo/results/checkpoint-1173"  # 你保存的模型路径
reward_model = GPT2LMHeadModel.from_pretrained(model_name)
tokenizer = GPT2Tokenizer.from_pretrained('gpt2-large')
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
# 将模型移至设备（CPU 或 GPU）
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
reward_model.to(device)

def get_reward_score(logits):
    """
    从模型的输出logits中提取奖励分数。
    这里使用 logits 的均值作为示例，你可以根据具体情况自定义。
    """
    return logits.mean().item()

def reward_fn(samples: List[str], **kwargs) -> List[float]:
    # 将样本tokenize并移至设备
    inputs = tokenizer(samples, return_tensors="pt", padding=True, truncation=True, max_length=512)
    inputs = {key: val.to(device) for key, val in inputs.items()}

    # 使用模型生成logits
    with torch.no_grad():
        outputs = reward_model(**inputs)
        logits = outputs.logits  # 获取logits

    # 计算每个样本的奖励分数
    rewards = [get_reward_score(logit) for logit in logits]

    return rewards

/pubshare/fwk/conda_envs/fdpo/lib/python3.10/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [8]:
samples = ['This film is very good and the music is fascinating', 'This film is a bit underwhelming in terms of plot, deep and difficult to understand']
print(reward_fn(samples))


[-5.5815887451171875, -5.6681694984436035]
